# 03 · 采样与引导  Sampling & Classifier-Free Guidance

> **能力标签**：GA3（扩散模型与流匹配 / Diffusion Models & Flow Matching）

## 目标 Objectives

完成本课后，你应该能够：

1. 实现 **ODE 欧拉法**（Euler method）和 **Heun 二阶法**（Heun's method）的单步更新，并比较精度与计算量。
2. 描述 **DDPM 反向采样**（reverse sampling）：从 $x_T \sim \mathcal{N}(0,I)$ 逐步去噪到 $x_0$，关键公式为均值预测与可选噪声项。
3. 推导**无分类器引导**（Classifier-Free Guidance, CFG）：通过线性组合条件与无条件分数（score），以引导强度 $w$ 在样本质量与多样性之间权衡：
   $\tilde{\epsilon} = (1+w)\, \epsilon_\theta(x_t, t, c) - w\, \epsilon_\theta(x_t, t, \emptyset)$
4. 实现 `euler_step` 和 `integrate`，并用解析速度场验证欧拉法在线性场下的精确性。

> 对应能力：**GA3**

## 概念讲解 Concepts

### 1. ODE 数值求解器 ODE Solvers

扩散模型与流匹配的推断均归结为求解常微分方程（ODE）：

$$\frac{dx}{dt} = v_\theta(x, t), \quad x(0) \sim p_0, \quad t \in [0, 1]$$

常用求解器：

**欧拉法**（Euler's method，1 阶）：

$$x_{t + \Delta t} = x_t + v_\theta(x_t, t) \cdot \Delta t$$

误差：$\mathcal{O}(\Delta t^2)$ per step，$\mathcal{O}(\Delta t)$ global。

**Heun 法**（Heun's method / RK2，2 阶）：

$$k_1 = v_\theta(x_t, t)$$
$$k_2 = v_\theta(x_t + k_1 \cdot \Delta t,\ t + \Delta t)$$
$$x_{t+\Delta t} = x_t + \frac{k_1 + k_2}{2} \cdot \Delta t$$

误差：$\mathcal{O}(\Delta t^3)$ per step，$\mathcal{O}(\Delta t^2)$ global；相同步数下精度更高。

---

### 2. DDPM 反向采样 DDPM Reverse Sampling

DDPM 的反向过程（reverse process）是一个逐步去噪的马尔科夫链：

$$p_\theta(x_{t-1}|x_t) = \mathcal{N}(x_{t-1};\ \mu_\theta(x_t, t),\ \sigma_t^2 I)$$

均值（mean）预测公式（利用 predict-$\epsilon$）：

$$\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right)$$

方差 $\sigma_t^2 = \beta_t$（Ho et al., 2020 选择固定方差）。

反向采样步骤（$t = T, T-1, \ldots, 1$）：
1. 用 $\epsilon_\theta(x_t, t)$ 预测噪声
2. 计算 $\mu_\theta(x_t, t)$
3. 采样 $x_{t-1} = \mu_\theta + \sigma_t \cdot z$（$t > 1$ 时 $z \sim \mathcal{N}(0,I)$，$t=1$ 时 $z=0$）

---

### 3. 无分类器引导 Classifier-Free Guidance (CFG)

Ho & Salimans (2022) 提出在不使用外部分类器的情况下实现条件生成引导。

训练时：以概率 $p_{\text{drop}}$ 将条件 $c$ 替换为空条件 $\emptyset$，使模型同时学习条件与无条件分数。

推断时，用**引导强度**（guidance scale） $w \geq 0$ 放大条件信号：

$$\tilde{\epsilon}_\theta(x_t, t, c) = (1 + w)\, \epsilon_\theta(x_t, t, c) - w\, \epsilon_\theta(x_t, t, \emptyset)$$

| $w$ | 效果 |
|-----|------|
| $w = 0$ | 标准条件采样 |
| $w > 0$ | 更高样本质量，多样性降低 |
| $w \gg 1$ | 可能过饱和（over-saturation） |

---

### 4. 实现摘要 Implementation Summary

```python
# 欧拉步 Euler step
def euler_step(x, v, dt):
    return x + v * dt

# 积分 Integration [0,1]
def integrate(x0, velocity_fn, n_steps):
    x = x0; dt = 1.0 / n_steps
    for i in range(n_steps):
        t = i * dt
        x = euler_step(x, velocity_fn(x, t), dt)
    return x
```

## 示例 Worked Example

In [ ]:
# Setup
import math
import tempfile
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
DEVICE = torch.device('cpu')
TMPDIR = tempfile.mkdtemp()
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
print('Tmp dir:', TMPDIR)

In [ ]:
# euler_step + integrate (mirrors w9-sampler)

def euler_step(x, v, dt):
    '''ODE 欧拉积分一步：x_{t+dt} = x + v * dt。
    One Euler step: x_{t+dt} = x + v * dt.
    x, v: any shape; dt: scalar.
    '''
    return x + v * dt


def integrate(x0, velocity_fn, n_steps):
    '''从 x0 出发，用 velocity_fn(x,t) 在 [0,1] 上欧拉积分 n_steps 步。
    Euler integration of velocity_fn(x, t) from t=0 to t=1 in n_steps steps.
    Args:
        x0: (B, D) starting point
        velocity_fn: callable (x, t) -> v
        n_steps: number of Euler steps
    Returns:
        x_1: (B, D) endpoint
    '''
    x = x0
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = i * dt
        x = euler_step(x, velocity_fn(x, t), dt)
    return x


# --- 单元测试 Unit Tests ---
torch.manual_seed(0)
B, D = 6, 3

# Test euler_step
x_t = torch.randn(B, D)
v_t = torch.randn(B, D)
dt_test = 0.1
x_next = euler_step(x_t, v_t, dt_test)
assert torch.allclose(x_next, x_t + v_t * dt_test, atol=1e-6), 'euler_step 错误'
print('\u2713 euler_step 通过 passed')

# Test integrate with constant velocity field: x(t) = x0 + v*t
# => x(1) = x0 + v exactly (Euler is exact for constant velocity)
torch.manual_seed(1)
x0_test = torch.randn(B, D)
v_const = torch.randn(B, D)
velocity_const = lambda x, t: v_const

x1_euler = integrate(x0_test, velocity_const, n_steps=100)
x1_exact = x0_test + v_const
err = (x1_euler - x1_exact).abs().max().item()
print(f'Euler vs exact (const field, n=100): max_err={err:.6f}')
assert err < 1e-4, f'欧拉误差过大: {err}'
print('\u2713 integrate 通过 passed (欧拉对常速度场精确 exact for const field)')

In [ ]:
# Heun 法实现与对比 Heun's method vs Euler

def heun_step(x, t, velocity_fn, dt):
    '''Heun（2 阶 Runge-Kutta）积分一步。
    Heun's method (RK2): 2nd-order accurate ODE step.
    '''
    k1 = velocity_fn(x, t)
    k2 = velocity_fn(x + k1 * dt, t + dt)
    return x + 0.5 * (k1 + k2) * dt


def integrate_heun(x0, velocity_fn, n_steps):
    '''Heun 法从 t=0 积分到 t=1。
    Heun integration from t=0 to t=1.
    '''
    x = x0
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = i * dt
        x = heun_step(x, t, velocity_fn, dt)
    return x


# Compare Euler vs Heun on a nonlinear velocity field
# True velocity: v(x, t) = -x  =>  x(t) = x0 * exp(-t) => x(1) = x0 / e
torch.manual_seed(42)
x0_cmp = torch.ones(1, 2) * 2.0
velocity_exp = lambda x, t: -x
x1_exact_exp = x0_cmp * math.exp(-1.0)

errors_euler, errors_heun = [], []
n_steps_list = [5, 10, 20, 50, 100]
for n in n_steps_list:
    xe = integrate(x0_cmp, velocity_exp, n)
    xh = integrate_heun(x0_cmp, velocity_exp, n)
    errors_euler.append((xe - x1_exact_exp).abs().max().item())
    errors_heun.append((xh - x1_exact_exp).abs().max().item())
    print(f'  n={n:4d}  Euler err={errors_euler[-1]:.6f}  Heun err={errors_heun[-1]:.6f}')

assert errors_heun[-1] < errors_euler[-1], 'Heun 应更精确'
print('\u2713 Heun 精度 > Euler (相同步数)')

In [ ]:
# DDPM 反向采样演示 DDPM reverse sampling demo

def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    betas = torch.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars


def ddpm_reverse_step(x_t, t_idx, eps_pred, betas, alphas, alpha_bars):
    '''DDPM 反向采样一步（predict-epsilon 参数化）。
    One DDPM reverse step using predicted epsilon.
    x_t: (B, D); t_idx: int; eps_pred: (B, D).
    Returns x_{t-1}: (B, D).
    '''
    beta_t = betas[t_idx]
    alpha_t = alphas[t_idx]
    alpha_bar_t = alpha_bars[t_idx]
    coef = beta_t / torch.sqrt(1.0 - alpha_bar_t)
    mu = (x_t - coef * eps_pred) / torch.sqrt(alpha_t)
    if t_idx == 0:
        return mu
    sigma_t = torch.sqrt(beta_t)
    return mu + sigma_t * torch.randn_like(x_t)


torch.manual_seed(42)
T_demo = 200
betas_d, alphas_d, abars_d = linear_beta_schedule(T_demo)
B_demo, D_demo = 4, 2

x_T = torch.randn(B_demo, D_demo)
print(f'x_T  norm: {x_T.norm(dim=1).mean().item():.3f}  (应接近 sqrt(D)={math.sqrt(D_demo):.3f})')

x_cur = x_T.clone()
for t_step in reversed(range(T_demo)):
    eps_dummy = torch.zeros_like(x_cur)
    x_cur = ddpm_reverse_step(x_cur, t_step, eps_dummy, betas_d, alphas_d, abars_d)

print(f'x_0 (eps=0) norm: {x_cur.norm(dim=1).mean().item():.4f}')
print('\u2713 DDPM 反向采样步骤形状正确 DDPM reverse sampling shape OK')

In [ ]:
# CFG 引导演示 Classifier-Free Guidance demo + visualization

def cfg_eps(eps_cond, eps_uncond, w):
    '''无分类器引导合成噪声预测。
    CFG: combine conditional and unconditional score with guidance scale w.
    eps_cond, eps_uncond: (B, D). w: guidance scale (w=0 => no guidance).
    '''
    return (1 + w) * eps_cond - w * eps_uncond


torch.manual_seed(0)
B_cfg, D_cfg = 8, 2

target_dir = torch.tensor([[1.0, 0.0]])
eps_cond_sim   = target_dir.expand(B_cfg, -1) + 0.1 * torch.randn(B_cfg, D_cfg)
eps_uncond_sim = torch.randn(B_cfg, D_cfg) * 0.5

guidance_scales = [0.0, 1.0, 3.0, 7.0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Euler vs Heun error curves
axes[0].loglog(n_steps_list, errors_euler, 'o-', label='Euler (1st order)',
               color='steelblue')
axes[0].loglog(n_steps_list, errors_heun, 's-', label='Heun (2nd order)',
               color='coral')
axes[0].set_xlabel('Number of Steps'); axes[0].set_ylabel('Max Error')
axes[0].set_title('欧拉 vs Heun 精度 Euler vs Heun Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# CFG effect
means_x, means_y = [], []
for w in guidance_scales:
    eps_guided = cfg_eps(eps_cond_sim, eps_uncond_sim, w)
    means_x.append(eps_guided[:, 0].mean().item())
    means_y.append(eps_guided[:, 1].mean().item())

axes[1].plot(guidance_scales, means_x, 'o-', label='Dim 0 (target dir)',
             color='steelblue')
axes[1].plot(guidance_scales, means_y, 's--', label='Dim 1 (noise)',
             color='coral')
axes[1].axhline(0, color='gray', linestyle=':', linewidth=0.8)
axes[1].set_xlabel('Guidance Scale $w$')
axes[1].set_ylabel('Mean eps value')
axes[1].set_title('CFG 引导效果 CFG Guidance Effect')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(TMPDIR, 'sampling_guidance.png')
plt.savefig(fig_path, dpi=80); plt.close()
print(f'图像已保存 Saved: {fig_path}')

cfg7 = cfg_eps(eps_cond_sim, eps_uncond_sim, 7.0)
cfg0 = cfg_eps(eps_cond_sim, eps_uncond_sim, 0.0)
assert cfg7[:, 0].mean().item() > cfg0[:, 0].mean().item(),     'w=7 应比 w=0 更强调条件方向'
print('\u2713 CFG 引导效果正确 CFG amplification verified')

## 动手 Exercise

实现 `integrate_heun_full`：给定速度场网络，用 Heun 法从 $x_0$ 积分到 $t=1$，与欧拉法 `integrate` 对比在同等步数下的误差（参考上方 `integrate_heun`）。


In [ ]:
# Exercise: Heun integration

def integrate_heun_full(x0, velocity_fn, n_steps):
    '''Heun 法从 t=0 积分到 t=1，返回 x_1。
    Full Heun integration [0,1] in n_steps steps.
    Args:
        x0: (B, D)
        velocity_fn: callable (x: Tensor, t: float) -> v: Tensor
        n_steps: int
    Returns:
        x_1: (B, D)
    '''
    # TODO: implement using heun_step defined above
    raise NotImplementedError('请实现 integrate_heun_full')


# Uncomment after implementation:
# torch.manual_seed(5)
# x0_ex2 = torch.ones(1, 2) * 3.0
# x1_heun_ex = integrate_heun_full(x0_ex2, velocity_exp, n_steps=20)
# x1_euler_ex = integrate(x0_ex2, velocity_exp, n_steps=20)
# x1_true = x0_ex2 * math.exp(-1.0)
# print(f'Heun  err: {(x1_heun_ex - x1_true).abs().max().item():.6f}')
# print(f'Euler err: {(x1_euler_ex - x1_true).abs().max().item():.6f}')
# assert (x1_heun_ex - x1_true).abs().max() < (x1_euler_ex - x1_true).abs().max()
# print('\u2713 Heun 精度高于 Euler')
print('提示：实现 integrate_heun_full 后取消注释运行验证')

## 延伸阅读 Further Reading

1. **Ho et al. "Denoising Diffusion Probabilistic Models" (NeurIPS 2020)**: <https://arxiv.org/abs/2006.11239> — DDPM 反向采样算法（Algorithm 2）的原始推导。
2. **Ho & Salimans "Classifier-Free Diffusion Guidance" (NeurIPS 2021 Workshop)**: <https://arxiv.org/abs/2207.12598> — CFG 原始论文；第 2 节给出引导公式与训练策略。
3. **Song et al. "Denoising Diffusion Implicit Models" (ICLR 2021)**: <https://arxiv.org/abs/2010.02502> — DDIM：确定性反向采样，可大幅减少采样步数。
4. **Lu et al. "DPM-Solver: A Fast ODE Solver for Diffusion Probabilistic Model Sampling" (NeurIPS 2022)**: <https://arxiv.org/abs/2206.00927> — 高阶 ODE 求解器，10~20 步达到高质量采样。
5. **Lipman et al. "Flow Matching for Generative Modeling" (ICLR 2023)**: <https://arxiv.org/abs/2210.02747> — 流匹配欧拉采样的原始论文。

## 关联练习 Related Assignments

```bash
python check.py w9-sampler
```

> 实现 `euler_step(x, v, dt)` 和 `integrate(x0, velocity_fn, n_steps)`。
> 关键：`euler_step` 仅一行 `x + v * dt`；`integrate` 在 `[0,1]` 上均匀分 `n_steps` 步，每步调用 `velocity_fn(x, t)` 得到速度。

> 能力标签：**GA3** · threshold ≥ 0.7